# Het trainen van het model door middel van k-means.

In [55]:
from sklearn.cluster import KMeans
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import time

# het importeren van de schone dataframe. 

#### Dit is al verwerkt in clean_kaggle_set.ipynb

In [56]:
df = pd.read_csv("./clean_kaggle_data.csv")
df.head(5)

,Unnamed: 0,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,0,-0.421694,-0.341057,-0.483092,29,0.533769,0.708833,1.27354,-1.022146,1.003082,-0.425061,-1.193553,0.294972,1.170156,-0.364173
1,1,1.087769,2.504900,0.367558,22,-0.401787,-0.525614,1.27354,0.506560,-1.598212,-0.425061,-1.296695,-0.640289,1.549404,2.109688
2,2,-0.400436,-0.324566,-0.484719,22,-1.265377,0.297351,1.27354,-0.410664,1.681681,0.506407,-1.246711,-0.484412,0.985233,-0.324085
3,3,-0.184929,0.106466,-0.258638,22,1.037530,-0.114132,1.27354,-0.716405,-0.354115,-0.425061,-1.207835,-0.328535,1.424735,0.007851
4,4,-0.377700,-0.204742,-0.195205,29,0.677700,-0.525614,1.27354,0.200819,1.116182,-0.425061,-1.290348,0.139096,1.730265,-0.281069


# het toepassen van k-means clustering

In [57]:
kmeans = KMeans(n_clusters=2, random_state=42)

### Ik maak gebruik van 2 clusters om het te sorteren op populair en niet populair.

In [58]:
columns_list = [
    "views", 
    "likes", 
    "comment_count", 
    "category_id", 
    "title_length", 
    "num_tags", 
    "year_published", 
    "month_published", 
    "day_published", 
    "hour_published", 
    "days_since_published", 
    "duration_seconds",
    "views_over_time",
    "engagement_rate"
]
df["popularity"] = kmeans.fit_predict(df[columns_list])
df["popularity"].value_counts()

popularity
1    404
0    138
Name: count, dtype: int64

## zoals je kan zien worden er binaire waardes toegewezen aan de rijen. De verdeling is erg ongelijk.

In [59]:
fig = px.scatter(df, x='views', y='likes', color='popularity',
                 title='views en likes met kleurtjes')
fig.show()

## in de scatterplot is te zien dat dit rampzalig slecht werkt. Ik denk dat dit komt omdat het uploaddatum mee rekent. Daarom ga ik het proberen zonder uploaddatum mee te rekenen.

In [60]:
columns_list = [
    "views", 
    "likes", 
    "comment_count",
    "title_length",
    "engagement_rate",
    "views_over_time",
    "num_tags"
]
df["popularity"] = kmeans.fit_predict(df[columns_list])
fig = px.scatter(df, x='views', y='likes', color='popularity',
                 title='views en likes met kleurtjes')
fig.show()

In [61]:
fig = px.box(df, x="popularity", y="views", title="views per cluster")
fig.show()

In [62]:
fig = px.box(df, x="popularity", y="likes", title="likes per cluster")
fig.show()

In [63]:
fig = px.box(df, x="popularity", y="comment_count", title="comment count per cluster")
fig.show()

In [64]:
fig = px.box(df, x="popularity", y="num_tags", title="hoeveelheid tags per cluster")
fig.show()

In [65]:
fig = px.box(df, x="popularity", y="views_over_time", title="views over tijd per cluster")
fig.show()

In [66]:
fig = px.box(df, x="popularity", y="engagement_rate", title="engagement rate per cluster")
fig.show()

In [67]:
fig = px.box(df, x="popularity", y="title_length", title="titel lengte per cluster")
fig.show()

# Conclusie
### In de eerste toepassing van het k-means algoritme heb ik niet het gewenste resultaat bereikt. Daarom heb ik een aantal factoren als uploaddatum en duratie verwijderd. Hierdoor werd de uitkomst beter. In de boxplots is te zien dat cluster 0 meer views, likes, comments, engagement rate en views/tijd heeft. Echter is er geen sterk verband tussen de clusters en de hoeveelheid gebruikte tags, en de lengte van de titel. Deze data column is dus niet nodig voor het trainen van het model.

## het model hertrainen met enkel de belanrijke waardes.

In [68]:
columns_list = [
    "views", 
    "likes", 
    "comment_count",
    "engagement_rate",
    "views_over_time"
]
start_time = time.time()
df["popularity"] = kmeans.fit_predict(df[columns_list])
end_time = time.time()
total_time = time.time() - start_time
print(f"latency van model trainen: {total_time} seconds")

latency van model trainen: 0.00457000732421875 seconds


In [69]:
num_samples = df.shape[0]
print(f"throughput van model trainen: {(total_time/num_samples) * 1000} ms/sample")

throughput van model trainen: 0.0084317478306619 ms/sample


## ter controle of het model nog steeds klopt

In [70]:
fig = px.scatter(df, x='views', y='likes', color='popularity',
                 title='views en likes met kleurtjes')
fig.show()

In [71]:
df['popularity'].value_counts()

popularity
0    492
1     50
Name: count, dtype: int64

In [72]:
import joblib
joblib.dump(kmeans, 'popularity_model.pkl')

['popularity_model.pkl']